# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SohailAkhtarChanna/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [21]:
import os
import duckdb

# Get the token from the Colab Secret.
HF_TOKEN = userdata.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN was not found. Check Colab Secrets."

# Connect to DuckDB.
con = duckdb.connect()

# Keep notebook output clean.
con.execute("SET enable_progress_bar = false")

# Store the Hugging Face token in the DuckDB session.
# The token is never written into the notebook source.
con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

# FlyRank warehouse.
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

# Development windows.
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Warehouse connection ready.")
print("Feature window: February 2026")
print("Outcome window: March 2026")

Warehouse connection ready.
Feature window: February 2026
Outcome window: March 2026


In [22]:
result = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {FEB}
""").df()

result

,rows,min_date,max_date
0,7355108,2026-02-01,2026-02-28


In [24]:
# Verify the warehouse grain:
# one row should represent one client × content × report_date.

grain_check = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM {FEB}
    GROUP BY
        client_hash_id,
        content_hash_id,
        report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate grain rows:", len(grain_check))
grain_check

Duplicate grain rows: 0


,client_hash_id,content_hash_id,report_date,row_count


In [25]:
# Verify the March 2026 outcome window

march_check = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {MAR}
""").df()

march_check

,rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [26]:
con.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW temp_feb AS SELECT * FROM {FEB};
    PRAGMA table_info('temp_feb')
""").df()

,cid,name,type,notnull,dflt_value,pk
0,0,report_date,DATE,False,None,False
1,1,client_hash_id,VARCHAR,False,None,False
2,2,content_hash_id,VARCHAR,False,None,False
3,3,client_has_gsc,BOOLEAN,False,None,False
4,4,client_has_ga4,BOOLEAN,False,None,False
5,5,gsc_data_available,BOOLEAN,False,None,False
6,6,ga4_data_available,BOOLEAN,False,None,False
7,7,gsc_impressions,BIGINT,False,None,False
8,8,gsc_clicks,BIGINT,False,None,False
9,9,gsc_sum_position,BIGINT,False,None,False


### Unit of analysis + time window

One row represents **one content item for one client** (`client_hash_id × content_hash_id`). The source warehouse table is at **client × content × day** grain, so the daily rows are aggregated separately into a feature window and an outcome window.

| Window        | Dates                   | Role                                                   |
| ------------- | ----------------------- | ------------------------------------------------------ |
| February 2026 | 2026-02-01 → 2026-02-28 | **Features** — information available before prediction |
| March 2026    | 2026-03-01 → 2026-03-31 | **Label/outcome** — future outcome being predicted     |

The two windows do not overlap. February information is used to make the prediction, while March is held out as the future outcome. June 2026 is excluded from development because it is the final/sealed month of the warehouse.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Fields: feature / label / context / excluded

**Features — February 2026 information available before the prediction**

* `gsc_impressions`
* `gsc_clicks`
* `gsc_avg_position`
* `ga4_pageviews`
* `ga4_sessions`
* `ga4_users`
* `ga4_engaged_sessions`
* `ga4_total_engagement_sec`
* `sessions_organic`
* `sessions_direct`
* `sessions_referral`
* `sessions_social`
* `sessions_paid`
* `sessions_ai`
* `ai_chatgpt`
* `ai_perplexity`
* `ai_gemini`
* `ai_copilot`
* `ai_claude`
* `ai_meta`
* `ai_other`
* `scroll_events`

**Label / proxy — March 2026 outcome**

The label will be calculated from March 2026 performance. It is kept completely separate from the February feature fields. The exact label definition will be verified from the observed March data rather than reusing the Week-2 `trend_direction` label.

**Context — used for grouping, joining, splitting, or reading**

* `report_date` — identifies the daily observation.
* `client_hash_id` — identifies the client and is used for grouping/splitting, not as a model feature.
* `content_hash_id` — identifies the content item and is used for grouping/joining, not as a model feature.
* `month` — identifies the warehouse partition/window.

**Excluded**

* `client_has_gsc` — availability metadata; not a performance feature.
* `client_has_ga4` — availability metadata; not a performance feature.
* `gsc_data_available` — measurement availability flag; used to interpret missingness rather than as a performance signal.
* `ga4_data_available` — measurement availability flag; used to filter/interprete GA4 observations rather than treating unavailable data as zero.
* Any March outcome fields used to construct the label — future information and therefore leakage if used as February features.
* June 2026 data — the final month is reserved as a sealed test month and is not used for development.


In [28]:
# Check missingness for the fields we plan to use.
# We calculate the percentage of NULL values in February.

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "ai_chatgpt",
    "ai_perplexity",
    "ai_gemini",
    "ai_copilot",
    "ai_claude",
    "ai_meta",
    "ai_other",
    "scroll_events"
]

missing_expr = ",\n".join(
    f"AVG(CASE WHEN {col} IS NULL THEN 1.0 ELSE 0 END) AS {col}_missing_pct"
    for col in feature_columns
)

missingness = con.sql(f"""
    SELECT
        {missing_expr}
    FROM {FEB}
""").df()

missingness.T

,0
gsc_impressions_missing_pct,0.012543
gsc_clicks_missing_pct,0.012543
gsc_avg_position_missing_pct,0.643543
ga4_pageviews_missing_pct,0.564550
ga4_sessions_missing_pct,0.564550
ga4_users_missing_pct,0.564550
ga4_engaged_sessions_missing_pct,0.564550
ga4_total_engagement_sec_missing_pct,0.564550
sessions_organic_missing_pct,0.564550
sessions_direct_missing_pct,0.564550


In [29]:
# Check whether missingness follows the warehouse availability flags.

availability_check = con.sql(f"""
    SELECT
        client_has_gsc,
        client_has_ga4,
        gsc_data_available,
        ga4_data_available,
        COUNT(*) AS rows,
        AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS gsc_impressions_missing_pct,
        AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS gsc_position_missing_pct,
        AVG(CASE WHEN ga4_sessions IS NULL THEN 1.0 ELSE 0 END) AS ga4_sessions_missing_pct,
        AVG(CASE WHEN sessions_ai IS NULL THEN 1.0 ELSE 0 END) AS sessions_ai_missing_pct
    FROM {FEB}
    GROUP BY
        client_has_gsc,
        client_has_ga4,
        gsc_data_available,
        ga4_data_available
    ORDER BY rows DESC
""").df()

availability_check

,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,rows,gsc_impressions_missing_pct,gsc_position_missing_pct,ga4_sessions_missing_pct,sessions_ai_missing_pct
0,True,False,False,<NA>,2669777,0.0,1.000000,1.0,1.0
1,True,True,False,False,1951314,0.0,1.000000,0.0,0.0
2,True,False,True,<NA>,1482546,0.0,0.000000,1.0,1.0
3,True,True,True,False,689992,0.0,0.000000,0.0,0.0
4,True,False,True,False,324565,0.0,0.000003,0.0,0.0
5,True,True,True,True,124680,0.0,0.000000,0.0,0.0
6,False,True,<NA>,False,91593,1.0,1.000000,0.0,0.0
7,True,True,False,True,19978,0.0,1.000000,0.0,0.0
8,False,True,<NA>,True,663,1.0,1.000000,0.0,0.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Missingness verification

The February 2026 missingness check shows that missing values are strongly patterned by data availability. When `ga4_data_available = FALSE`, GA4 session data is missing, while when `ga4_data_available = TRUE`, `ga4_sessions` is effectively complete. Similarly, GSC position is missing when `gsc_data_available = FALSE` and is effectively complete when GSC data is available.

Therefore, missing values are treated as **data-availability indicators rather than zeros**. The availability flags are retained as context/quality checks, and unavailable measurements will not be interpreted as zero performance.


In [ ]:
# Inspect March outcome variables and their availability.

march_summary = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS contents,

        SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END)
            AS rows_with_gsc,

        SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END)
            AS rows_with_ga4,

        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position,

        SUM(ga4_sessions) AS total_ga4_sessions,
        SUM(scroll_events) AS total_scroll_events

    FROM {MAR}
""").df()

march_summary

In [ ]:
# Check how many client-content pairs have usable GSC data
# in BOTH the February feature window and March outcome window.

pair_availability = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS feb_gsc_days
    FROM {FEB}
    GROUP BY client_hash_id, content_hash_id
),
mar AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS mar_gsc_days
    FROM {MAR}
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    COUNT(*) AS total_pairs,
    SUM(CASE WHEN feb_gsc_days > 0 THEN 1 ELSE 0 END) AS pairs_with_feb_gsc,
    SUM(CASE WHEN mar_gsc_days > 0 THEN 1 ELSE 0 END) AS pairs_with_mar_gsc,
    SUM(
        CASE
            WHEN feb_gsc_days > 0 AND mar_gsc_days > 0
            THEN 1 ELSE 0
        END
    ) AS pairs_with_both
FROM feb
JOIN mar USING (client_hash_id, content_hash_id)
""").df()

pair_availability

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Final data-limit checks

print("Feature window:", "2026-02-01 to 2026-02-28")
print("Outcome window:", "2026-03-01 to 2026-03-31")
print("Windows overlap:", False)
print("June 2026 used for development:", False)
print("Final ML-04 data contract complete.")

### Data limits

This dataset cannot provide a complete view of every client and content item because history and measurement availability differ across clients.

GSC and GA4 data are only available where the corresponding measurement system is available. Missing measurements therefore cannot automatically be interpreted as zero performance.

The February and March windows also have different numbers of usable client-content pairs, so changes in availability must not be confused with changes in performance.

The warehouse contains daily observations, but this contract aggregates them into separate feature and outcome windows. The March outcome must never be used to construct February features.

The final June 2026 month is excluded from development because it is the sealed final month. The warehouse should therefore not be treated as evidence about performance beyond its available observation period.

This contract describes measurable search and analytics behavior only. It cannot establish why a page's performance changed or prove causation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.